In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
from statsmodels.stats.multitest import multipletests
import time

EA_GENO  = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
EA_META  = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR  = r"C:\Users\user\Desktop\ai causal\FTND\european_american"

os.makedirs(OUT_DIR, exist_ok=True)
print("Imports done.")

Imports done.


In [2]:
meta_df = pd.read_csv(EA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers = smokers[smokers["cpd"] >= 0].dropna(subset=["cpd"])

print("Total EA samples:", len(meta_df))
print("Smokers with valid CPD:", len(smokers))
print("\nCPD distribution:")
print(smokers["cpd"].describe())

Total EA samples: 1595
Smokers with valid CPD: 863

CPD distribution:
count    863.000000
mean      27.753187
std        7.732058
min        7.000000
25%       20.000000
50%       30.000000
75%       30.000000
max       65.000000
Name: cpd, dtype: float64


In [3]:
def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(
    manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"]
)

encoded_df = pd.read_csv(EA_GENO)
all_sample_ids = encoded_df.columns[1:].tolist()
probe_id_array = encoded_df["probe_id"].to_numpy()

smoker_id_set = set(smokers["sample_id"].tolist())
smoker_cols_cpd = [s for s in all_sample_ids if s in smoker_id_set]
print("Smoker columns found:", len(smoker_cols_cpd))

smokers_aligned = smokers.set_index("sample_id").reindex(smoker_cols_cpd).reset_index()
Y_cpd = smokers_aligned["cpd"].values.astype(np.float64)
age_std = ((smokers_aligned["age"].astype(float) - smokers_aligned["age"].astype(float).mean()) /
           smokers_aligned["age"].astype(float).std()).values
gender_binary = (smokers_aligned["gender"] == "Male").astype(np.float64).values

keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])
X_snp = encoded_df[smoker_cols_cpd].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp.T
probe_ids_auto = probe_id_array[keep_mask]

del encoded_df, X_snp
gc.collect()

print("X_auto shape:", X_auto.shape)
print("Y_cpd mean:", Y_cpd.mean().round(2))

Smoker columns found: 863
X_auto shape: (863, 233721)
Y_cpd mean: 27.75


In [4]:
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
print("Monomorphic excluded:", (~valid_mask).sum())
print("Informative retained:", valid_mask.sum())

X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]
del X_float
gc.collect()

pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)
print("Explained variance ratio:", pca.explained_variance_ratio_)

X_conf = np.hstack([
    pcs,
    age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("X_conf shape:", X_conf.shape)

Monomorphic excluded: 124802
Informative retained: 108919
Explained variance ratio: [0.02598186 0.00744911 0.00517623 0.0050741  0.00502152 0.00497708
 0.00491197 0.00484885 0.00482651 0.00479767]
X_conf shape: (863, 12)


In [5]:
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

print("Running single test iteration...")
start = time.time()
theta_test, pvals_test = doubleml_scan(X_std, Y_cpd, X_conf, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.001, 0.0001, 1e-5, 1e-6]:
    print(f"Raw p < {thresh}: {(pvals_test < thresh).sum()} SNPs")

Running single test iteration...
Completed in 11.1s
Raw p < 0.001: 3295 SNPs
Raw p < 0.0001: 1522 SNPs
Raw p < 1e-05: 213 SNPs
Raw p < 1e-06: 69 SNPs


In [6]:
n_repeats = 30
threshold = 1e-5
n_snps = X_std.shape[1]

significant_counts = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold}) for EA CPD...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y_cpd, X_conf, random_state=rep)
    significant_counts += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction = significant_counts / n_repeats

stability_df_ea_cpd = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

stability_df_ea_cpd.to_csv(os.path.join(OUT_DIR, "cpd_stability_results.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction >= thresh).sum()} SNPs")

Running 30 stability repeats (p<1e-05) for EA CPD...
  Repeat 5/30 — 1.0 min
  Repeat 10/30 — 2.1 min
  Repeat 15/30 — 3.2 min
  Repeat 20/30 — 4.2 min
  Repeat 25/30 — 5.3 min
  Repeat 30/30 — 6.4 min

Total: 6.4 min
  >= 50% stability: 62 SNPs
  >= 70% stability: 46 SNPs
  >= 80% stability: 13 SNPs
  >= 90% stability: 6 SNPs
  >= 100% stability: 4 SNPs


In [7]:
shortlist_ea_cpd = stability_df_ea_cpd[stability_df_ea_cpd["stability_fraction"] >= 0.70].copy()
shortlist_ea_cpd["core_name"] = shortlist_ea_cpd["probe_id"].map(strip_suffix)

pos_lookup_manifest = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_ea_cpd = shortlist_ea_cpd.merge(pos_lookup_manifest, on="core_name", how="left")
shortlist_ea_cpd = shortlist_ea_cpd[~shortlist_ea_cpd["Chr"].isin(non_autosomal)].copy()
print("Shortlist size:", len(shortlist_ea_cpd))

# LD pruning
encoded_df_ld = pd.read_csv(EA_GENO)
probe_rows = encoded_df_ld[encoded_df_ld["probe_id"].isin(set(shortlist_ea_cpd["probe_id"]))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(shortlist_ea_cpd["probe_id"].tolist())
X_shortlist = probe_rows[smoker_cols_cpd].to_numpy(dtype=np.float64).T
probe_id_to_idx = {pid: i for i, pid in enumerate(shortlist_ea_cpd["probe_id"].tolist())}
del encoded_df_ld, probe_rows
gc.collect()

def get_geno(pid):
    return X_shortlist[:, probe_id_to_idx[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained = greedy_ld_prune(shortlist_ea_cpd, r2_thresh=0.2)
shortlist_pruned_ea_cpd = shortlist_ea_cpd[
    shortlist_ea_cpd["probe_id"].isin(retained)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned_ea_cpd)} SNPs")
shortlist_pruned_ea_cpd.to_csv(os.path.join(OUT_DIR, "cpd_shortlist_ld_pruned.csv"), index=False)
print(shortlist_pruned_ea_cpd[["probe_id", "Chr", "MapInfo"]].to_string())

Shortlist size: 46
After LD pruning: 44 SNPs
                       probe_id Chr      MapInfo
0     exm24035-0_B_R_1921346506   1   17982446.0
1     exm48976-0_T_R_1919174050   1   40703169.0
2     exm85854-0_B_F_1919187493   1  114523753.0
3    exm840569-0_B_R_1921062879  10   91099466.0
4    exm842015-0_B_F_1920960758  10   94267953.0
5    exm842539-0_B_F_1921030758  10   95093631.0
6    exm857621-0_T_R_1920983209  10  116049170.0
7    exm883775-0_B_R_1918227141  11    5776338.0
8    exm892742-0_T_R_1918353158  11   16810755.0
9    exm999027-0_B_F_1922568481  12   49230563.0
10  exm1014552-0_T_R_1922437488  12   57351053.0
11  exm1018255-0_T_R_1922577212  12   59313238.0
12  exm1039373-0_T_R_1922441190  12  113552611.0
13  exm1111853-0_B_R_1922709948  14   71443826.0
14  exm1164448-0_T_R_1922802638  15   56719803.0
15  exm1283758-0_T_F_1918697769  17    5324787.0
16  exm1315010-0_B_F_1918742545  17   36482977.0
17  exm1337922-0_T_R_1918724846  17   50008372.0
18  exm1361294-0_B_R_191

In [8]:
import pandas as pd
import numpy as np
import json
import os
import re
import gc

EA_GENO  = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
EA_META  = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR  = r"C:\Users\user\Desktop\ai causal\FTND\european_american"
os.makedirs(OUT_DIR, exist_ok=True)

def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# metadata
meta_df = pd.read_csv(EA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers = smokers[smokers["cpd"] >= 0].dropna(subset=["cpd"])

# genotype column order
encoded_df = pd.read_csv(EA_GENO)
all_sample_ids = encoded_df.columns[1:].tolist()
smoker_cols_cpd = [s for s in all_sample_ids if s in set(smokers["sample_id"])]
smokers_aligned = smokers.set_index("sample_id").reindex(smoker_cols_cpd).reset_index()
Y_cpd = smokers_aligned["cpd"].values.astype(np.float64)

# reload shortlist
shortlist_pruned_ea_cpd = pd.read_csv(os.path.join(OUT_DIR, "cpd_shortlist_ld_pruned.csv"))
pruned_ids_ea_cpd = shortlist_pruned_ea_cpd["probe_id"].tolist()
col_names_ea_cpd = pruned_ids_ea_cpd + ["CPD"]

# build PC input
probe_rows = encoded_df[encoded_df["probe_id"].isin(set(pruned_ids_ea_cpd))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(pruned_ids_ea_cpd)
X_pc_ea_cpd = probe_rows[smoker_cols_cpd].to_numpy(dtype=np.float64).T
X_pc_full_ea_cpd = np.hstack([X_pc_ea_cpd, Y_cpd.reshape(-1, 1)])

print("PC input shape:", X_pc_full_ea_cpd.shape)  # expect (863, 45)

np.save(os.path.join(OUT_DIR, "cpd_pc_input.npy"), X_pc_full_ea_cpd)
with open(os.path.join(OUT_DIR, "cpd_pc_col_names.json"), "w") as f:
    json.dump(col_names_ea_cpd, f)
print("Saved.")

PC input shape: (863, 45)
Saved.


In [9]:
import pandas as pd
import numpy as np
import json
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
import time

EA_GENO  = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
EA_META  = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR  = r"C:\Users\user\Desktop\ai causal\FTND\european_american"
os.makedirs(OUT_DIR, exist_ok=True)

def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# metadata
meta_df = pd.read_csv(EA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers = smokers[smokers["cpd"] >= 0].dropna(subset=["cpd"])
print("Smokers with valid CPD:", len(smokers))

# manifest
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"])

# genotype
encoded_df = pd.read_csv(EA_GENO)
all_sample_ids = encoded_df.columns[1:].tolist()
probe_id_array = encoded_df["probe_id"].to_numpy()
smoker_cols_cpd = [s for s in all_sample_ids if s in set(smokers["sample_id"])]
smokers_aligned = smokers.set_index("sample_id").reindex(smoker_cols_cpd).reset_index()
Y_cpd = smokers_aligned["cpd"].values.astype(np.float64)
age_std = ((smokers_aligned["age"].astype(float) - smokers_aligned["age"].astype(float).mean()) /
           smokers_aligned["age"].astype(float).std()).values
gender_binary = (smokers_aligned["gender"] == "Male").astype(np.float64).values

keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])
X_snp = encoded_df[smoker_cols_cpd].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp.T
probe_ids_auto = probe_id_array[keep_mask]
del X_snp
gc.collect()

# standardize
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]
del X_float
gc.collect()
print("X_std shape:", X_std.shape)

# PCA + confounders
pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)
X_conf = np.hstack([pcs, age_std.reshape(-1,1), gender_binary.reshape(-1,1)]).astype(np.float64)
print("X_conf shape:", X_conf.shape)

# DoubleML
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])
    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D
    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid
    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()
    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

# stability selection
n_repeats = 30
threshold = 1e-5
significant_counts = np.zeros(X_std.shape[1], dtype=np.int32)

print(f"Running {n_repeats} stability repeats...")
start = time.time()
for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y_cpd, X_conf, random_state=rep)
    significant_counts += (pvals_rep < threshold).astype(np.int32)
    if (rep+1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction = significant_counts / n_repeats
stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

stability_df.to_csv(os.path.join(OUT_DIR, "cpd_stability_results.csv"), index=False)
print(f"Done in {(time.time()-start)/60:.1f} min")
for t in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction >= t).sum()} SNPs")

Smokers with valid CPD: 863
X_std shape: (863, 108919)
X_conf shape: (863, 12)
Running 30 stability repeats...
  Repeat 5/30 — 0.8 min
  Repeat 10/30 — 1.6 min
  Repeat 15/30 — 2.4 min
  Repeat 20/30 — 3.2 min
  Repeat 25/30 — 4.0 min
  Repeat 30/30 — 4.8 min
Done in 4.8 min
  >= 50%: 62 SNPs
  >= 70%: 46 SNPs
  >= 80%: 13 SNPs
  >= 90%: 6 SNPs
  >= 100%: 4 SNPs


In [11]:
# LD pruning on >=70% stable SNPs
shortlist_ea_cpd = stability_df[stability_df["stability_fraction"] >= 0.70].copy()
shortlist_ea_cpd["core_name"] = shortlist_ea_cpd["probe_id"].map(strip_suffix)

pos_lookup_m = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_ea_cpd = shortlist_ea_cpd.merge(pos_lookup_m, on="core_name", how="left")
shortlist_ea_cpd = shortlist_ea_cpd[~shortlist_ea_cpd["Chr"].isin(non_autosomal)].copy()
print("Shortlist before LD pruning:", len(shortlist_ea_cpd))

# load genotype vectors
probe_rows_ld = encoded_df[encoded_df["probe_id"].isin(set(shortlist_ea_cpd["probe_id"]))].copy()
probe_rows_ld = probe_rows_ld.set_index("probe_id").reindex(shortlist_ea_cpd["probe_id"].tolist())
X_ld = probe_rows_ld[smoker_cols_cpd].to_numpy(dtype=np.float64).T
probe_id_to_idx_ld = {pid: i for i, pid in enumerate(shortlist_ea_cpd["probe_id"].tolist())}

def get_geno(pid):
    return X_ld[:, probe_id_to_idx_ld[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained = greedy_ld_prune(shortlist_ea_cpd)
shortlist_pruned_ea_cpd = shortlist_ea_cpd[
    shortlist_ea_cpd["probe_id"].isin(retained)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned_ea_cpd)} SNPs")
shortlist_pruned_ea_cpd.to_csv(os.path.join(OUT_DIR, "cpd_shortlist_ld_pruned.csv"), index=False)

# check perfect correlations before building PC input
pruned_ids_ea_cpd = shortlist_pruned_ea_cpd["probe_id"].tolist()
probe_rows_pc = encoded_df[encoded_df["probe_id"].isin(set(pruned_ids_ea_cpd))].copy()
probe_rows_pc = probe_rows_pc.set_index("probe_id").reindex(pruned_ids_ea_cpd)
X_check = probe_rows_pc[smoker_cols_cpd].to_numpy(dtype=np.float64).T

corr_matrix = np.corrcoef(X_check.T)
to_remove = set()
for i in range(len(pruned_ids_ea_cpd)):
    for j in range(i+1, len(pruned_ids_ea_cpd)):
        if abs(corr_matrix[i,j]) > 0.99 and pruned_ids_ea_cpd[j] not in to_remove:
            to_remove.add(pruned_ids_ea_cpd[j])
print(f"Perfectly correlated SNPs to remove: {len(to_remove)}")

final_ids = [p for p in pruned_ids_ea_cpd if p not in to_remove]
col_names_ea_cpd = final_ids + ["CPD"]

probe_rows_final = encoded_df[encoded_df["probe_id"].isin(set(final_ids))].copy()
probe_rows_final = probe_rows_final.set_index("probe_id").reindex(final_ids)
X_pc_ea_cpd = probe_rows_final[smoker_cols_cpd].to_numpy(dtype=np.float64).T
X_pc_full_ea_cpd = np.hstack([X_pc_ea_cpd, Y_cpd.reshape(-1, 1)])

print("Final PC input shape:", X_pc_full_ea_cpd.shape)

np.save(os.path.join(OUT_DIR, "cpd_pc_input.npy"), X_pc_full_ea_cpd)
with open(os.path.join(OUT_DIR, "cpd_pc_col_names.json"), "w") as f:
    json.dump(col_names_ea_cpd, f)
print("Saved successfully.")

Shortlist before LD pruning: 46
After LD pruning: 44 SNPs
Perfectly correlated SNPs to remove: 27
Final PC input shape: (863, 18)
Saved successfully.
